In [ ]:
import copy
import json

import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import Divider, Size
import numpy as np
import pandas as pd
from datafold import EDMDWindowPrediction, TSCDataFrame, DMDStandard

from dynamics_analysis_with_koopman.config.configs import PROJECT_ROOT
from dynamics_analysis_with_koopman.utils.checkpoint_registry import (
    checkpoint_path_for,
    load_run_from_checkpoint,
    load_train_tsc_from_checkpoint,
)
from dynamics_analysis_with_koopman.plots.io_utils import save_figure

# importing from dynamics_analysis_with_koopman.plots runs setup_seaborn_style()
# reset to plain matplotlib defaults so this notebook's own plots
plt.rcdefaults()

WINDOWS_SIZE = 40
OFFSET = WINDOWS_SIZE
TEMPORAL_RESOLUTION = 0.2
STEPWISE_TRANSFORM = False
REMOVE_SPECTRAL_POLLUTION = False
# scenarios whose time series are too short for window predictions, use the single prediction instead
SINGLE_PREDICTION_SCENARIOS = ['2022wheelchairB', '2022wheelchairU']
FIGURE_FORMAT = 'svg'  # change this alone (e.g. to 'eps' or 'png') to switch the saved plot format

PLOTS_ROOT = PROJECT_ROOT / 'papers' / 'clustering'
PLOTS_DIR = 'window_reconstruction'

# absolute width (in cm) and font size (in points) used for grid plots that will
# be placed side by side in the paper, so scenarios with a different number of
# grid cells still end up the same physical size and font size
PAPER_GRID_WIDTH_CM = 5.0
PAPER_FONT_SIZE = 14

# small fixed breathing-room margin (in inches) used between measured text
# elements (titles, tick labels, legend, ...) when computing paddings below;
# actual paddings are measured from the real rendered text, so they scale
# automatically with font size
LAYOUT_MARGIN_IN = 0.12

# fixed width (in inches) of each grid's colorbar, and the gap between a grid
# and its own colorbar
CBAR_WIDTH_IN = 0.14
CBAR_GAP_IN = LAYOUT_MARGIN_IN

# extra breathing room (in inches) added between one grid's colorbar and the
# next grid's own y-tick labels, on top of the usual margins
CBAR_TO_NEXT_GRID_EXTRA_GAP_IN = 0.2

TS_ROW_HEIGHT_CM = 3.0

RUN_REGISTRY = PROJECT_ROOT / 'papers' / 'clustering' / 'juelich_run_registry.json'
with open(RUN_REGISTRY) as f:
    registry = json.load(f)

In [ ]:
# Helper functions
def _get_window_predictions(series_id, edmd, train_tsc, use_stepwise_transform=False):
    # reconstruct the predictions in windows
    window_size = edmd.n_samples_ic_ + WINDOWS_SIZE

    # adapt_model overwrites `reconstruct` on the model it gets, adapt a copy so `edmd` keeps its
    # single prediction for later series
    edmd_wp = EDMDWindowPrediction(window_size=window_size, offset=OFFSET).adapt_model(copy.copy(edmd))
    if use_stepwise_transform:
        edmd_wp.stepwise_transform = True

    # passes feature name to .predict() to reduce computational effort via `qois`
    X_truth = TSCDataFrame.from_single_timeseries(train_tsc.loc[series_id])
    window_predictions = edmd_wp.reconstruct(X=X_truth)  # , qois=[cell_str])

    return window_predictions


def _load_scenario(scenario):
    # load the model + run config/metrics/style from the checkpoint
    edmd, run_config, run_metrics, style = load_run_from_checkpoint(experiment=scenario, registry=registry)
    style = style.with_(path=PLOTS_ROOT, fig_filetype=FIGURE_FORMAT, dpi=300)

    train_tsc, train_tsc_predicted = load_train_tsc_from_checkpoint(
        checkpoint_path_for(experiment=scenario, registry=registry), with_predictions=True
    )

    # remove spectral pollution
    if REMOVE_SPECTRAL_POLLUTION:
        dmd: DMDStandard = edmd.dmd_model
        dmd.residual_filter = {'keep_n': 25}
        (dmd.eigenvalues_, dmd.eigenvectors_right_, dmd.eigenvectors_left_) = dmd._remove_spectral_pollution(
            eigenvalues=dmd.eigenvalues_,
            eigenvectors_right=dmd.eigenvectors_right_,
            eigenvectors_left=dmd.eigenvectors_left_,
            G=dmd._G,
            G_dash=dmd._G_dash,
            R=dmd._R,
        )

        edmd._koopman_modes = edmd._compute_koopman_modes(
            inverse_map=edmd._inverse_map,
        )

        print(f'Reduced the eigenvalues to 25: {len(dmd.eigenvalues_)=}')

    return edmd, style, train_tsc, train_tsc_predicted


def _get_series_predictions(scenario, series_id, edmd, train_tsc, train_tsc_predicted):
    """
    Ground truth, single prediction and window predictions of one series; the
    window predictions are None for scenarios that only use the single prediction
    and for series shorter than one window.
    """
    true_data = train_tsc.loc[series_id]

    # apply window prediction settings
    if scenario in SINGLE_PREDICTION_SCENARIOS or len(true_data) < edmd.n_samples_ic_ + WINDOWS_SIZE:
        window_predictions = None
        X_truth = TSCDataFrame.from_single_timeseries(train_tsc.loc[series_id])
        if STEPWISE_TRANSFORM:
            edmd.stepwise_transform = True
            predictions = edmd.reconstruct(X=X_truth)  # , qois=[cell_str])

        else:
            # execute prediction, because it may change after removal of Koopman triples
            predictions = edmd.reconstruct(X=X_truth)  # , qois=[cell_str])

    else:
        predictions = train_tsc_predicted.loc[series_id]
        window_predictions = _get_window_predictions(
            series_id,
            edmd,
            train_tsc,
            use_stepwise_transform=STEPWISE_TRANSFORM,
        )

    return true_data, predictions, window_predictions


def _get_mae(true_data, window_predictions, predictions, observable):
    """
    MAE of the (window) predictions over all cells of `observable`, together with
    the MAE of a baseline that predicts each cell's temporal mean over the same
    evaluated time steps, and the range, mean and std of the ground truth.
    """
    observable_columns = [col for col in true_data.columns if observable in col]
    true_data = true_data.copy()[observable_columns]

    mean_value = np.mean(true_data.values)
    std = np.std(true_data.values)
    value_range = np.max(true_data.values) - np.min(true_data.values)

    if window_predictions is not None:
        prediction_windows = [window_predictions.loc[idx][observable_columns] for idx in window_predictions.ids]
    else:  # use only single predictions
        prediction_windows = [predictions.loc[0][observable_columns]]

    abs_errors = []
    evaluated_truth = []
    for prediction in prediction_windows:
        # align timesteps and calculate absolute error per cell
        truth = true_data.loc[true_data.index.intersection(prediction.index)]
        prediction = prediction.loc[truth.index]
        abs_errors.append(np.abs((prediction.values - truth.values).real))
        evaluated_truth.append(truth.values.real)

    evaluated_truth = np.concatenate(evaluated_truth)
    baseline_mae = np.abs(evaluated_truth - evaluated_truth.mean(axis=0)).mean()

    return np.concatenate(abs_errors).mean(), baseline_mae, value_range, mean_value, std


def _bbox_height_in(fig, renderer, artist):
    return artist.get_window_extent(renderer=renderer).transformed(fig.dpi_scale_trans.inverted()).height


def _bbox_width_in(fig, renderer, artist):
    return artist.get_window_extent(renderer=renderer).transformed(fig.dpi_scale_trans.inverted()).width


def _left_overhang_in(fig, renderer, ax):
    """How far (inches) the leftmost y-tick label extends past `ax`'s own left edge.

    Using the label's rendered position (rather than just its own text width)
    captures whatever tick length / tick pad the axes applies -- a rotated grid
    gets extra tick pad from `format_grid_axes`, which a plain text-width
    measurement misses, clipping the label off the saved figure.
    """
    labels = ax.get_yticklabels()
    if not labels:
        return 0.0
    ax_left_px = ax.get_window_extent(renderer=renderer).x0
    label_left_px = min(t.get_window_extent(renderer=renderer).x0 for t in labels)
    return max(ax_left_px - label_left_px, 0.0) / fig.dpi


def _pin_suptitle_top(fig, fig_height_in):
    """
    `fig.suptitle` positions itself at a fixed figure-fraction (~0.98) by default,
    independent of any axes layout -- pin it explicitly to the top of the figure
    so it always lands inside the padding reserved for it, at any font size.
    """
    if fig._suptitle is not None:
        fig._suptitle.set_verticalalignment('top')
        fig._suptitle.set_y((fig_height_in - LAYOUT_MARGIN_IN) / fig_height_in)


def _fix_grid_axes_size(fig, axs, width_in, height_in, bottom_labels=None, colorbars=None):
    """
    Forces each axes in `axs` (arranged in a single row) to have an identical,
    absolute plot-area size of `width_in` x `height_in` inches, independent of
    the number of grid cells. All paddings (for y-tick labels, the suptitle, the
    top x-tick labels from `ax.xaxis.tick_top()`, one `bottom_labels` text artist
    per axes placed underneath it, and -- if given -- one `colorbars` axes per
    grid placed to its right) are measured from the actual rendered text, so
    this scales correctly with font size. This overrides any prior
    tight_layout-based positioning, since the locator set here governs the
    final draw position.
    """
    n = len(axs)

    # pass 1: left/gap paddings only depend on y-tick label widths (and, if
    # present, the previous grid's colorbar tick label widths), which are
    # independent of the figure's width -- fix the figure's final width first
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()

    def cbar_label_w(i):
        if colorbars is None:
            return 0.0
        return max((_bbox_width_in(fig, renderer, t) for t in colorbars[i].get_yticklabels()), default=0.0)

    pad_left_in = _left_overhang_in(fig, renderer, axs[0]) + LAYOUT_MARGIN_IN
    gaps_in = [
        cbar_label_w(i)
        + LAYOUT_MARGIN_IN
        + (CBAR_TO_NEXT_GRID_EXTRA_GAP_IN if colorbars is not None else 0.0)
        + _left_overhang_in(fig, renderer, axs[i + 1])
        + LAYOUT_MARGIN_IN
        for i in range(n - 1)
    ]
    pad_right_in = LAYOUT_MARGIN_IN + (cbar_label_w(n - 1) + LAYOUT_MARGIN_IN if colorbars is not None else 0.0)

    horiz = [Size.Fixed(pad_left_in)]
    main_nx = []
    cbar_nx = []
    for i in range(n):
        main_nx.append(len(horiz))
        horiz.append(Size.Fixed(width_in))
        if colorbars is not None:
            horiz.append(Size.Fixed(CBAR_GAP_IN))
            cbar_nx.append(len(horiz))
            horiz.append(Size.Fixed(CBAR_WIDTH_IN))
        if i < n - 1:
            horiz.append(Size.Fixed(gaps_in[i]))
    horiz.append(Size.Fixed(pad_right_in))
    fig_width_in = sum(size.fixed_size for size in horiz)

    # pass 2: apply the final width, then measure vertical paddings -- a
    # wrap=True suptitle reflows based on the actual figure width, so its
    # height can only be measured correctly once the width is final
    fig.set_size_inches(fig_width_in, fig.get_size_inches()[1])
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()

    suptitle_h = _bbox_height_in(fig, renderer, fig._suptitle) if fig._suptitle is not None else 0.0
    xtick_h = max((_bbox_height_in(fig, renderer, t) for ax in axs for t in ax.get_xticklabels()), default=0.0)
    pad_top_in = suptitle_h + LAYOUT_MARGIN_IN + xtick_h + LAYOUT_MARGIN_IN

    pad_bottom_in = LAYOUT_MARGIN_IN
    if bottom_labels is not None:
        label_h = max((_bbox_height_in(fig, renderer, lbl) for lbl in bottom_labels), default=0.0)
        pad_bottom_in += label_h + LAYOUT_MARGIN_IN

    vert = [Size.Fixed(pad_bottom_in), Size.Fixed(height_in), Size.Fixed(pad_top_in)]

    divider = Divider(fig, (0, 0, 1, 1), horiz, vert, aspect=False)
    for i, ax in enumerate(axs):
        ax.set_axes_locator(divider.new_locator(nx=main_nx[i], ny=1))
    if colorbars is not None:
        for i, cbar_ax in enumerate(colorbars):
            cbar_ax.set_axes_locator(divider.new_locator(nx=cbar_nx[i], ny=1))

    fig_height_in = sum(size.fixed_size for size in vert)
    fig.set_size_inches(fig_width_in, fig_height_in)
    _pin_suptitle_top(fig, fig_height_in)

    if bottom_labels is not None:
        # center each label under its own axes column (not its colorbar), using
        # the same left/width/gap layout computed above
        x_in = pad_left_in
        for i, label in enumerate(bottom_labels):
            center_x_in = x_in + width_in / 2
            label.set_verticalalignment('bottom')
            label.set_position((center_x_in / fig_width_in, LAYOUT_MARGIN_IN / fig_height_in))
            x_in += width_in
            if colorbars is not None:
                x_in += CBAR_GAP_IN + CBAR_WIDTH_IN
            if i < n - 1:
                x_in += gaps_in[i]


def _fix_column_axes_size(fig, axs, width_in, height_in, legend):
    """
    Forces each axes in `axs` (stacked in a single column, top to bottom, sharing
    the x-axis) to have an identical, absolute size of `width_in` x `height_in`
    inches, independent of scenario-specific content. All paddings (the
    suptitle, each row's own title, the shared x-axis label + tick labels, and
    -- if present -- the legend) are measured from the actual rendered text, so
    this scales correctly with font size and with however many legend entries
    there are. This overrides any prior tight_layout-based positioning, since
    the locator set here governs the final draw position.
    """
    n = len(axs)

    # pass 1: left padding only depends on y-tick/y-label widths, which are
    # independent of the figure's width -- fix the figure's final width first
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()

    ytick_w = max((_bbox_width_in(fig, renderer, t) for ax in axs for t in ax.get_yticklabels()), default=0.0)
    ylabel_w = max((_bbox_width_in(fig, renderer, ax.yaxis.label) for ax in axs), default=0.0)
    # extra margin beyond the usual 2x: fig.align_ylabels() (called after this
    # function) re-centers each y-label on the same vertical line, which can push
    # a label slightly further left than its own individually-measured width, and
    # without this room labels get clipped off the left edge of the saved SVG
    pad_left_in = ytick_w + ylabel_w + 3 * LAYOUT_MARGIN_IN
    pad_right_in = LAYOUT_MARGIN_IN
    fig_width_in = pad_left_in + width_in + pad_right_in

    # pass 2: apply the final width, then measure vertical paddings -- a
    # wrap=True suptitle reflows based on the actual figure width, so its
    # height can only be measured correctly once the width is final
    fig.set_size_inches(fig_width_in, fig.get_size_inches()[1])
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()

    suptitle_h = _bbox_height_in(fig, renderer, fig._suptitle) if fig._suptitle is not None else 0.0
    top_title_h = _bbox_height_in(fig, renderer, axs[0].title)
    row_title_heights = [_bbox_height_in(fig, renderer, ax.title) for ax in axs[1:]]
    # every axes shows its own x-tick labels and its own x-axis label (axs are
    # not linked via sharex), so the gap above a row's title must also clear
    # both of those from the row above it, not just that row's own title
    upper_xtick_h = max(
        (_bbox_height_in(fig, renderer, t) for ax in axs[:-1] for t in ax.get_xticklabels()), default=0.0
    )
    upper_xlabel_h = max((_bbox_height_in(fig, renderer, ax.xaxis.label) for ax in axs[:-1]), default=0.0)
    vgap_in = (
        upper_xtick_h
        + LAYOUT_MARGIN_IN
        + upper_xlabel_h
        + LAYOUT_MARGIN_IN
        + (max(row_title_heights) if row_title_heights else 0.0)
        + 2 * LAYOUT_MARGIN_IN
    )
    pad_top_in = suptitle_h + 2 * LAYOUT_MARGIN_IN + top_title_h

    tick_h = max((_bbox_height_in(fig, renderer, t) for t in axs[-1].get_xticklabels()), default=0.0)
    xlabel_h = _bbox_height_in(fig, renderer, axs[-1].xaxis.label)
    pad_bottom_in = tick_h + 2 * LAYOUT_MARGIN_IN + xlabel_h
    if legend is not None:
        legend_h = _bbox_height_in(fig, renderer, legend)
        pad_bottom_in += 2 * LAYOUT_MARGIN_IN + legend_h

    horiz = [Size.Fixed(pad_left_in), Size.Fixed(width_in), Size.Fixed(pad_right_in)]
    vert = [Size.Fixed(pad_bottom_in)]
    for i in range(n):
        vert.append(Size.Fixed(height_in))
        vert.append(Size.Fixed(vgap_in if i < n - 1 else pad_top_in))

    divider = Divider(fig, (0, 0, 1, 1), horiz, vert, aspect=False)
    for i, ax in enumerate(axs):
        # axs is ordered top to bottom; Divider's ny counts cells from the bottom
        row_from_bottom = n - 1 - i
        ny = 1 + 2 * row_from_bottom
        ax.set_axes_locator(divider.new_locator(nx=1, ny=ny))

    fig_height_in = sum(size.fixed_size for size in vert)
    fig.set_size_inches(fig_width_in, fig_height_in)
    _pin_suptitle_top(fig, fig_height_in)


# create the plot
def _plot_predictions(
    train_tsc,
    train_tsc_predicted,
    predicted_data,
    true_data,
    window_predictions,
    temporal_resolution,
    cell_str,
    series_id,
    ax,
):

    # ground truth and one-shot prediction
    true_time_values = TSCDataFrame.from_single_timeseries(train_tsc.loc[series_id]).time_values()
    ax.plot(
        true_time_values * temporal_resolution,
        true_data[cell_str],
        color='black',
        alpha=0.8,
        linestyle='dotted',
        label='Ground truth',
    )

    if window_predictions is not None:
        # get the window predictions
        window_predictions = window_predictions[cell_str]
        for idx in window_predictions.ids:
            # Write the series data to a dictionary structure
            timesteps = window_predictions.loc[idx].index.values
            predictions = window_predictions.loc[idx][cell_str].values
            line = ax.plot(
                (timesteps) * temporal_resolution,
                predictions,
                linestyle='solid',
                label=f'Prediction [{timesteps[0] * temporal_resolution:.1f}, {timesteps[-1] * temporal_resolution:.1f}]',
            )
            color = line[0].get_c()
            ax.plot(timesteps[0] * temporal_resolution, predictions[0], 'o', color=color, markersize=5, alpha=0.8)

    else:
        pred_time_values = TSCDataFrame.from_single_timeseries(train_tsc_predicted.loc[series_id]).time_values()
        predicted_data = predicted_data[cell_str].values
        plot_time_values = pred_time_values[: len(predicted_data)] * temporal_resolution
        line = ax.plot(
            plot_time_values,
            predicted_data,
            linestyle='solid',
            label=f'Prediction [{plot_time_values[0]:.1f}, {plot_time_values[-1]:.1f}]',
        )
        color = line[0].get_c()
        ax.plot(plot_time_values[0], predicted_data[0], 'o', color=color, markersize=5, alpha=0.8)

    t_max = true_time_values[-1] * temporal_resolution
    ax.set_xlim(0, t_max)
    ax.grid()

In [ ]:
## median NMAE of the window reconstruction per scenario, over all of its series
# NMAE = MAE / range of the ground truth, the fraction of the ground truth range
nmae_rows = []
for scenario, run in registry['runs'].items():
    if 'learning_run_dir' not in run:
        print(f'Skip {scenario}: no learning run')
        continue

    edmd, _, train_tsc, train_tsc_predicted = _load_scenario(scenario)

    for series_id in train_tsc.ids:
        true_data, predictions, window_predictions = _get_series_predictions(
            scenario, series_id, edmd, train_tsc, train_tsc_predicted
        )
        for observable in ['density', 'velocity_x', 'velocity_y']:
            mae, _, value_range, _, _ = _get_mae(true_data, window_predictions, predictions, observable)
            nmae_rows.append(
                {
                    'Scenario': scenario,
                    'Series': series_id,
                    'Observable': observable.replace('_', ' '),
                    'Windowed': window_predictions is not None,
                    'NMAE': mae / value_range,
                }
            )

nmae_df = pd.DataFrame(nmae_rows)
nmae_summary = nmae_df.groupby(['Scenario', 'Observable'])['NMAE'].median().unstack('Observable')
nmae_summary.insert(0, 'Series', nmae_df.groupby('Scenario')['Series'].nunique())
nmae_summary.columns = pd.MultiIndex.from_tuples(
    [('', col) if col == 'Series' else ('NMAE median', col) for col in nmae_summary.columns],
    names=[None, 'Scenario'],
)
nmae_summary.index.name = None

print(
    nmae_summary.to_latex(
        index=True,
        column_format='lr|ccc',
        float_format='%.3f',
        multicolumn_format='c',
        caption='Median NMAE of the window reconstruction per scenario, over all of its series.',
        label='tab:reconstruction-nmae',
        position='ht',
    )
)

print(f'Max NMAE over all single trials and observable:')
print(nmae_df.groupby(['Observable'])['NMAE'].max())

In [ ]:
## Plot the window reconstructions for the spot check
validation_time_series = {
    # for spot-check
    '2009tjunction': [100240150240240, '(13_4)', 12, 'ko-240-150-240.txt'],
    '2009unidirOpenC': [100050180180180, '(7_18)', 16, 'uo-050-180-180.txt'],
    '2013crossing90E': [10090511, '(13_22)', 60, 'CROSSING_90_E_1.txt'],
    '2013crossing120C': [10012030707, '(8_14)', 24, 'crossing_120_c_07.txt'],
    '2013unidirectionalC': [1005000101, '(10_7)', 32, 'traj_UNI_CORR_500_01.txt'],
    '2022wheelchairU': [1001100100, '(10_7)', 8, '1AM100.txt'],
}

## Plot the window predictions for each scenario
width_in = PAPER_GRID_WIDTH_CM * 0.6
height_in = TS_ROW_HEIGHT_CM * 0.4

stats_per_trial = []

# get the window predictions
for scenario, (series_id, measurement_cell, _, traj_file_name) in validation_time_series.items():
    print(f'\n\n\nProcess scenario {scenario}')

    edmd, style, train_tsc, train_tsc_predicted = _load_scenario(scenario)
    true_data, predictions, window_predictions = _get_series_predictions(
        scenario, series_id, edmd, train_tsc, train_tsc_predicted
    )

    # create a figure per series
    fig, axs = plt.subplots(nrows=3, ncols=1, figsize=(5, 10), sharex=False)
    for observable, ax in zip(['density', 'velocity_x', 'velocity_y'], axs):
        mae, baseline_mae, value_range, mean, std = _get_mae(true_data, window_predictions, predictions, observable)
        stats_per_trial.append(
            {
                'Scenario': scenario,
                'Trial': traj_file_name.replace('_', r'\_'), # escape the file names
                'Observable': f'{observable.replace("_", " ")}',
                'Mean $\pm$ Std': f'{mean:.3f} $\pm$ {std:.3f}',
                'NMAE': f'{mae / value_range:.3f}',
            }
        )
        print(f'MAE for {observable}: {mae}, NMAE: {mae / value_range}')

        # visualize the time series
        cell_str = f'{measurement_cell}_{observable}'
        _plot_predictions(
            train_tsc,
            train_tsc_predicted,
            predictions,
            true_data,
            window_predictions,
            style.temporal_resolution,
            cell_str,
            series_id,
            ax,
        )

        # configure the plot
        ax.set_ylabel(observable.replace('_', ' '), fontsize=PAPER_FONT_SIZE)
        ax.set_title(f'Values for {observable.replace("_", " ")}', fontsize=PAPER_FONT_SIZE)
        ax.tick_params(labelsize=PAPER_FONT_SIZE)

        # configure full figure
        ax.set_xlabel('Time in seconds', fontsize=PAPER_FONT_SIZE)

    fig.suptitle(
        f'Window predictions for {scenario} in {measurement_cell}, series id {series_id} ',
        fontsize=PAPER_FONT_SIZE,
        wrap=True,
    )

    # single shared legend, placed in the reserved bottom margin so it can't
    # overlap the x-axis label (no window predictions -> nothing to label)
    handles, labels = axs[-1].get_legend_handles_labels()
    legend = None
    if handles:
        legend = fig.legend(
            handles,
            labels,
            loc='lower center',
            ncol=2,
            fontsize=PAPER_FONT_SIZE - 5,
            bbox_to_anchor=(0.5, 0.0),
            bbox_transform=fig.transFigure,
            borderaxespad=0.3,
        )

    # layouting
    _fix_column_axes_size(fig, axs, width_in, height_in, legend)

    # align the y-axis labels of the three subplots onto the same vertical line,
    # regardless of how wide each label ("density" vs "velocity x") is
    fig.align_ylabels(axs)

    # save the plot
    name = f'{scenario}_window_prediction_w{WINDOWS_SIZE}_o{OFFSET}_{series_id}_{measurement_cell.replace("(", "").replace(")", "")}'
    save_figure(fig, style, PLOTS_DIR, name, close_after=False, bbox_inches=None)
    print(f'Stored the window prediction plot to {style.path / PLOTS_DIR / f"{name}.{style.fig_filetype}"}')
    plt.show()

    print(stats_per_trial)

In [ ]:
## get latex table for the appendix reporting
df = pd.DataFrame(stats_per_trial).pivot(index=['Scenario', 'Trial'], columns='Observable')

latex_table = df.to_latex(
    index=True,
    column_format='ll|ccc|ccc',
    multicolumn_format='c',
    float_format='%.3f',
    caption='\\acl{MAE} per scenario across all cells',
    label='tab:reconstruction-mae',
    position='ht',
    bold_rows=False,
)

print(latex_table)